In [2]:
from random import random
from math import log, floor, ceil
from statistics import mean, stdev
import matplotlib.pyplot as plt
from pandas import DataFrame

In [3]:
def random_uniform(N):
    x = random()
    return x * N

def random_exp(N):
    x = random()
    return log(1 - x) * (-1/N)

def random_triangle(N):
    x = random()
    return N - N * (1 - x)**0.5

def construct_function(function_name, N):
    def generator():
          return function_name(N)
    return generator

In [4]:
def generate(N, func):
    nums = []
    for x in range(N):
        num = func()
        nums.append(num)
    return nums

In [5]:
exp_service_function = construct_function(random_exp, 1)
uniform_service_function = construct_function(random_uniform, 2)
triangle_service_function = construct_function(random_triangle, 3)

In [6]:
def model_of_mass_service(intensity, order_amount, service_function):
    arrival_function = construct_function(random_exp, intensity)

    # arrival_time_distance - время с прибытия предыдущего покупателя
    # arrival_time - время с прибытия первого покупателя
    # service_time - время обслуживания
    # service_delay - длина очереди в минутах на момент прибытия покупателя
    # queue_pop_time - время окончания ожидания в очереди
    # queue_length - количество заявок в очереди
    df = DataFrame(columns=['arrival_time_distance',
                            'arrival_time',
                            'service_time',
                            'service_delay',
                            'queue_pop_time',
                            'queue_length'])

    for ind in range(order_amount):
        service_time = service_function()
        if ind != 0:
            arrival_time_distance = arrival_function()
            arrival_time = arrival_time_distance + df.loc[ind - 1, 'arrival_time']
            queue_time = df.loc[ind - 1, 'service_time'] + df.loc[ind - 1, 'service_delay']
            delay_time = max(queue_time - arrival_time_distance, 0)
            queue_pop_time = arrival_time + delay_time
            queue_length  = df.loc[ind - 1, 'queue_length']
            for client in range(ind - int(queue_length), ind):
                if df.loc[client, 'queue_pop_time'] < arrival_time:
                    queue_length -= 1
            if delay_time > 0:
                queue_length += 1
        else:
            arrival_time_distance = 0
            arrival_time = 0
            delay_time = 0
            queue_pop_time = 0
            queue_length = 0

        df.loc[ind] = [arrival_time_distance,
                       arrival_time,
                       service_time,
                       delay_time,
                       queue_pop_time,
                       queue_length]

    # print(df)

    QM = max(df['queue_length'])
    QA = sum(df['service_delay']) / (df.loc[order_amount-1, "arrival_time"] + df.loc[order_amount-1, "service_delay"] + df.loc[order_amount-1, "service_time"])
    QZ = (df["queue_length"] == 0).sum()
    QT = sum(df['service_delay']) / order_amount
    QX = sum(df['service_delay']) / (order_amount - QZ)
    FR = sum(df['service_time']) / (df.loc[order_amount-1, "arrival_time"] + df.loc[order_amount-1, "service_delay"] + df.loc[order_amount-1, "service_time"])
    FT = sum(df['service_time']) / order_amount
    print('QM:', QM)
    print('QA:', QA)
    print('QZ:', QZ)
    print('QT:', QT)
    print('QX:', QX)
    print('FR:', FR)
    print('FT:', FT)


In [7]:
model_of_mass_service(0.55, 1000, uniform_service_function)

QM: 7.0
QA: 0.40436924228780796
QZ: 458
QT: 0.7467343433904771
QX: 1.3777386409418397
FR: 0.551145009549891
FT: 1.0177799490650945


In [8]:
model_of_mass_service(0.55, 47500, uniform_service_function)

QM: 9.0
QA: 0.44548900058157204
QZ: 21141
QT: 0.8086130175866094
QX: 1.4571538501219299
FR: 0.5515826286687358
FT: 1.0011849747893204


In [9]:
model_of_mass_service(0.75, 1000, uniform_service_function)

QM: 15.0
QA: 1.901250959263037
QZ: 199
QT: 2.48432912066711
QX: 3.101534482730474
FR: 0.7754880139360286
FT: 1.0133157047801586


In [10]:
model_of_mass_service(0.75, 47500, uniform_service_function)

QM: 21.0
QA: 1.4764062857816218
QZ: 12150
QT: 1.9798751850229894
QX: 2.6603697677112303
FR: 0.7457647900675992
FT: 1.0000778349010049


In [11]:
model_of_mass_service(0.55, 1000, exp_service_function)

QM: 7.0
QA: 0.6181899420261001
QZ: 449
QT: 1.1305068464313637
QX: 2.0517365633963047
FR: 0.5444131372323296
FT: 0.9955884706101241


In [12]:
model_of_mass_service(0.55, 47500, exp_service_function)

QM: 26.0
QA: 0.7009739931584504
QZ: 21227
QT: 1.2726305896127816
QX: 2.300839379081457
FR: 0.5503517249297014
FT: 0.9991732175909384


In [13]:
model_of_mass_service(0.75, 1000, exp_service_function)

QM: 19.0
QA: 2.593019920421701
QZ: 232
QT: 3.4938920555805524
QX: 4.549338614037178
FR: 0.765083656644609
FT: 1.030890541469651


In [14]:
model_of_mass_service(0.75, 47500, exp_service_function)

QM: 24.0
QA: 2.1584608793616664
QZ: 12098
QT: 2.9001019520005835
QX: 3.891159898311613
FR: 0.7438123467069956
FT: 0.9993841719498912


In [15]:
model_of_mass_service(0.55, 1000, triangle_service_function)

QM: 7.0
QA: 0.5419585753866062
QZ: 455
QT: 0.9974836463128404
QX: 1.8302452225923678
FR: 0.5526504832396784
FT: 1.0171622780675191


In [16]:
model_of_mass_service(0.55, 47500, triangle_service_function)

QM: 11.0
QA: 0.49973003453000403
QZ: 21327
QT: 0.9122652261122859
QX: 1.6556221388581203
FR: 0.5503184989930398
FT: 1.0046152827092418


In [17]:
model_of_mass_service(0.75, 1000, triangle_service_function)

QM: 12.0
QA: 2.023638076252406
QZ: 233
QT: 2.6557841938649362
QX: 3.4625608785722766
FR: 0.7900415047692193
FT: 1.0368354724521691


In [18]:
model_of_mass_service(0.75, 47500, triangle_service_function)

QM: 20.0
QA: 1.6462019531133996
QZ: 12050
QT: 2.2076291051353554
QX: 2.9580361775438466
FR: 0.7485988884722649
FT: 1.0039039810017194


In [20]:
order_amounts = [2000, 4000, 8000, 16000, 32000]

print("UNIFORM 0.55")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.55, amount, uniform_service_function)
    print("=" * 50)

print("UNIFORM 0.75")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.75, amount, uniform_service_function)
    print("=" * 50)

print("EXP 0.55")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.55, amount, exp_service_function)
    print("=" * 50)

print("EXP 0.75")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.75, amount, exp_service_function)
    print("=" * 50)

print("TRIANGLE 0.55")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.55, amount, triangle_service_function)
    print("=" * 50)

print("TRIANGLE 0.75")
print("=" * 10)
for amount in order_amounts:
    print(amount)
    model_of_mass_service(0.75, amount, triangle_service_function)
    print("=" * 50)

UNIFORM 0.55
2000
QM: 7.0
QA: 0.4549458479083371
QZ: 881
QT: 0.7976480621882124
QX: 1.425644436440058
FR: 0.5601906257570749
FT: 0.9821717664761755
4000
QM: 9.0
QA: 0.42392866478831837
QZ: 1794
QT: 0.7756544113838876
QX: 1.406444988910041
FR: 0.5336836445734137
FT: 0.976471061242108
8000
QM: 8.0
QA: 0.4424752042197873
QZ: 3649
QT: 0.8095632109603028
QX: 1.4885096960888125
FR: 0.5458297805169937
FT: 0.9986632144331443
16000
QM: 9.0
QA: 0.451016427102354
QZ: 7329
QT: 0.8259616101573055
QX: 1.5240901582881892
FR: 0.541586780521136
FT: 0.991826156207014
32000
QM: 11.0
QA: 0.44690984731130995
QZ: 14365
QT: 0.807506519194603
QX: 1.4652797626440202
FR: 0.5515595881974624
FT: 0.9965946507405815
UNIFORM 0.75
2000
QM: 14.0
QA: 1.6230062606335913
QZ: 463
QT: 2.1463724712877554
QX: 2.7929375033022192
FR: 0.7607261435519405
FT: 1.0060353384412497
4000
QM: 15.0
QA: 1.5126112364788444
QZ: 977
QT: 2.040609946989036
QX: 2.700112400911725
FR: 0.7424122330386007
FT: 1.0015619023375537
8000
QM: 15.0
QA: 1

In [21]:
print("=" * 10)
for experiment in range(10):
    print("EXPERIMENT", experiment)
    model_of_mass_service(0.75, 32000, exp_service_function)
    print("=" * 50)

EXPERIMENT 0
QM: 26.0
QA: 2.115200964491028
QZ: 8115
QT: 2.837174317622551
QX: 3.801112755449932
FR: 0.7458846794062078
FT: 1.000474608250113
EXPERIMENT 1
QM: 30.0
QA: 2.334641887416219
QZ: 7830
QT: 3.094492922095392
QX: 4.09697035610478
FR: 0.7519036888520142
FT: 0.9966242170978231
EXPERIMENT 2
QM: 26.0
QA: 2.0805551904423703
QZ: 8278
QT: 2.786773129056235
QX: 3.759242059261425
FR: 0.7435179441126049
FT: 0.9958956326381267
EXPERIMENT 3
QM: 31.0
QA: 2.1440644654795036
QZ: 7880
QT: 2.847886546041074
QX: 3.7782906083463668
FR: 0.7567131647701717
FT: 1.005115879610074
EXPERIMENT 4
QM: 28.0
QA: 2.3379528600635022
QZ: 8076
QT: 3.1188759250674862
QX: 4.171711653659904
FR: 0.7444006702688817
FT: 0.9930453982903035
EXPERIMENT 5
QM: 23.0
QA: 2.1574699662167265
QZ: 7975
QT: 2.8730042700114056
QX: 3.826686228527158
FR: 0.7531708706706602
FT: 1.002963267794406
EXPERIMENT 6
QM: 34.0
QA: 2.319352310270472
QZ: 8200
QT: 3.13378506438944
QX: 4.213492523548827
FR: 0.7409715061022281
FT: 1.00116115549971